In [0]:
%sql

SELECT COUNT(DISTINCT PERSON_ID) FROM 4_prod.rde.rde_all_diagnosis
WHERE Diagnosis_code = 'C22.0'
AND Diagnosis_date BETWEEN '2006-01-01' AND '2026-12-31'
LIMIT 100

In [0]:
%sql

SELECT * --COUNT(DISTINCT PERSON_ID)
FROM 4_prod.rde.rde_all_problems
WHERE code_text ILIKE '%Hepatocellular carcinoma%'
LIMIT 100

In [0]:
%sql

SELECT COUNT(DISTINCT PERSON_ID) FROM 4_prod.raw.mill_diagnosis
WHERE DIAGNOSIS_DISPLAY ILIKE '%Hepatocellular carcinoma%' OR DIAGNOSIS_DISPLAY ILIKE '%HCC%'


In [0]:
%sql

WITH d AS (
SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
WHERE Diagnosis_code = 'C22.0'
),
p AS (
SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_problems
WHERE code_text ILIKE '%Hepatocellular carcinoma%'
)
SELECT COALESCE(d.PERSON_ID, p.PERSON_ID) AS PERSON_ID
FROM d FULL OUTER JOIN p
ON d.PERSON_ID = p.PERSON_ID

In [0]:
%sql
SELECT * FROM 4_prod.rde.rde_all_diagnosis
WHERE Diagnosis_code = 'C22.0'
AND Diagnosis_date BETWEEN '2006-01-01' AND '2026-12-31'

In [0]:
%sql
WITH d AS (
SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
WHERE Diagnosis_code = 'C22.0'
),
p AS (
SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_problems
WHERE code_text ILIKE '%Hepatocellular carcinoma%'
),
hcc AS (
    SELECT COALESCE(d.PERSON_ID, p.PERSON_ID) AS PERSON_ID
    FROM d FULL OUTER JOIN p
    ON d.PERSON_ID = p.PERSON_ID
),
t AS (
    SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_procedures
    WHERE (Procedure_code IN (
        "J10.1",  -- TAE/TACE
        "J12.3",  -- SIRT
        "J12.4"  -- RFA
    ) OR
    code_text ILIKE '%radiofrequency%'
     OR code_text ILIKE '%RFA%'
     OR code_text ILIKE '%SIRT%'
     OR code_text ILIKE '%Y-90%'
     OR code_text ILIKE '%radioembol%')
    AND procedure_date BETWEEN '2006-01-01' AND '2026-12-31'
),
t2 AS (
    SELECT * FROM 4_prod.rde.rde_all_procedures
     WHERE       
)
SELECT COUNT(DISTINCT hcc.PERSON_ID) FROM hcc
INNER JOIN t
ON hcc.PERSON_ID = t.PERSON_ID


In [0]:
%sql
CREATE VIEW 5_projects.dar067.rde_patient_demographics AS 
WITH d AS (
    SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
    WHERE Diagnosis_code = 'C22.0'
),
p AS (
    SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_problems
    WHERE code_text ILIKE '%Hepatocellular carcinoma%'
),
pid AS (
    SELECT COALESCE(d.PERSON_ID, p.PERSON_ID) AS PERSON_ID
    FROM d FULL OUTER JOIN p
    ON d.PERSON_ID = p.PERSON_ID
)
SELECT de.* EXCEPT (NHS_Number, MRN, Date_of_Birth) FROM 4_prod.rde.rde_patient_demographics AS de
INNER JOIN pid ON pid.PERSON_ID = de.PERSON_ID



In [0]:
for t in ["rde_all_diagnosis", "rde_all_problems", "rde_all_procedures", "rde_pathology"]:
    cols = ", ".join([
        f"t.{col}" for col in spark.table(f"4_prod.rde.{t}").columns
        if col not in {"NHS_Number", "MRN", "Date_of_Birth", "BlobContents"}
    ])

    q = f"""
    CREATE OR REPLACE VIEW 5_projects.dar067.{t} AS
    WITH pid AS (
    SELECT DISTINCT PERSON_ID FROM 5_projects.dar067.rde_patient_demographics
    )
    SELECT 
    {cols}
    FROM 4_prod.rde.{t} AS t
    INNER JOIN pid
    ON t.PERSON_ID = pid.PERSON_ID
    """
    spark.sql(q)

In [0]:
for t in ["rde_all_diagnosis", "rde_all_problems", "rde_all_procedures", "rde_pathology"]:
    q = f"""SELECT COUNT(PERSON_ID), COUNT(DISTINCT PERSON_ID) FROM 5_projects.dar067.{t}"""
    display(spark.sql(q))

In [0]:
%sql
SELECT im.* FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN 5_projects.dar067.rde_patient_demographics AS pd
ON im.PersonID = pd.PERSON_ID
WHERE ExaminationModality = "CT" AND ExaminationBodyPart IN ('ABDOMEN', 'CHEST_TO_PELVIS')
LIMIT 100

In [0]:
%sql
SELECT DISTINCT ExamCOde FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN 5_projects.dar067.rde_patient_demographics AS pd
ON im.PersonID = pd.PERSON_ID
WHERE ExaminationModality = "CT" AND ExaminationBodyPart IN ('ABDOMEN', 'CHEST_TO_PELVIS')
LIMIT 100

In [0]:
%sql
SELECT * FROM 4_prod.pacs_dlt.pacs_examcode_dict
WHERE modality_id = 77477000 -- CT

In [0]:
%sql

CREATE OR REPLACE VIEW 5_projects.dar067.imaging_metadata AS
WITH pid AS (
    SELECT PERSON_ID FROM 5_projects.dar067.rde_patient_demographics
)
SELECT im.* EXCEPT (NHSNumber, MRN) FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN pid
ON pid.PERSON_ID = im.PersonID
WHERE ExaminationModality = "CT" AND ExaminationBodyPart IN ('ABDOMEN', 'CHEST_TO_PELVIS')
